# Module 3: Light-Matter Interaction & Cavity QED
## Notebook 3.1: Two-Level Atoms & Optical Bloch Equations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_3_light_matter_cavity_qed/01_two_level_atoms_and_bloch_equations.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** **Light-Matter Interaction & Cavity QED**
  * **3.1 Two-Level Atoms & Optical Bloch Equations** $\leftarrow$ *(You are here)*
  * **3.2** Rabi Oscillations, Detuning & Pulse Control
  * **3.3** The Jaynes-Cummings Hamiltonian & RWA
  * **3.4** The Quantum Rabi Model & Dressed States
  * **3.5** Quantum Collapse and Revival of Atomic Inversion
  * **3.6** Resonance Fluorescence & The Mollow Triplet
  * **3.7** Electromagnetically Induced Transparency (EIT) & Dark States

---

### Learning Objectives
1. **Semiclassical Dipole Coupling:** Derive the two-level Hamiltonian in the electric dipole approximation and transform into the rotating wave frame.
2. **The Optical Bloch Equations (OBEs):** Formulate the equations of motion for the density matrix $\hat{\rho}$ incorporating spontaneous emission ($T_1$) and dephasing ($T_2$).
3. **The 3D Bloch Vector Representation:** Map the density matrix to the geometric Bloch sphere $\vec{u} = (u, v, w)$ and visualize dissipative trajectories.
4. **Saturation & Power Broadening:** Calculate the analytical steady-state solutions, explain the fundamental limit $\rho_{ee} \le 0.5$ for CW excitation, and compute power-broadened Lorentzian absorption profiles.
5. **Numerical Simulation:** Implement a Runge-Kutta numerical solver for arbitrary detuning, drive amplitudes, and dephasing rates.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Physical Motivation: Semiclassical Atom-Field Coupling

Consider a two-level atom with ground state $|g\rangle$ and excited state $|e\rangle$, separated by transition frequency $\omega_0 = (E_e - E_g)/\hbar$. The atomic dipole operator is:
$$\hat{\vec{d}} = \vec{d}_{ge} |g\rangle\langle e| + \vec{d}_{eg} |e\rangle\langle g| = \vec{d}_{eg} (\hat{\sigma}_+ + \hat{\sigma}_-)$$
where $\hat{\sigma}_+ = |e\rangle\langle g|$ and $\hat{\sigma}_- = |g\rangle\langle e|$ are the atomic raising and lowering operators, and $\vec{d}_{eg} = \langle e|\hat{\vec{d}}|g\rangle$ is the transition dipole matrix element (assumed real for simplicity).

In the presence of a classical monochromatic laser field $\vec{E}(t) = \vec{E}_0 \cos(\omega_L t)$, the semiclassical Hamiltonian under the electric dipole approximation is:
$$\hat{H}(t) = \hat{H}_0 + \hat{H}_{\text{int}}(t) = \frac{\hbar \omega_0}{2} \hat{\sigma}_z - \hat{\vec{d}} \cdot \vec{E}_0 \cos(\omega_L t)$$
where $\hat{\sigma}_z = |e\rangle\langle e| - |g\rangle\langle g|$.

### The Rotating Wave Approximation (RWA)
Transforming into the reference frame rotating at laser frequency $\omega_L$ via unitary transformation $\hat{U}(t) = \exp(i \frac{\omega_L t}{2} \hat{\sigma}_z)$ and neglecting fast-oscillating terms at $\pm 2\omega_L$ (the Rotating Wave Approximation, valid when $|\omega_L - \omega_0| \ll \omega_0$ and $\Omega \ll \omega_0$), we obtain the time-independent RWA Hamiltonian:
$$\hat{H}_{\text{RWA}} = -\frac{\hbar \Delta}{2} \hat{\sigma}_z + \frac{\hbar \Omega}{2} (\hat{\sigma}_+ + \hat{\sigma}_-) = \frac{\hbar}{2} \begin{pmatrix} -\Delta & \Omega \\ \Omega & \Delta \end{pmatrix}$$
where:
* $\Delta = \omega_L - \omega_0$ is the **laser detuning**.
* $\Omega = -\frac{\vec{d}_{eg} \cdot \vec{E}_0}{\hbar}$ is the **Rabi frequency** measuring the atom-field coupling strength.

---

## 2. Dissipative Dynamics & The Optical Bloch Equations

A real atom in free space interacts with the continuum of vacuum electromagnetic modes, leading to **spontaneous emission** at rate $\Gamma = 1/T_1$, as well as collisions and environmental noise causing pure dephasing at rate $\gamma_{\text{pure}}$.

The full dynamics are governed by the Lindblad master equation:
$$\frac{d\hat{\rho}}{dt} = -\frac{i}{\hbar} [\hat{H}_{\text{RWA}}, \hat{\rho}] + \mathcal{D}[\hat{L}_{\text{decay}}](\hat{\rho}) + \mathcal{D}[\hat{L}_{\text{deph}}](\hat{\rho})$$
with jump operators:
* $\hat{L}_{\text{decay}} = \sqrt{\Gamma} \hat{\sigma}_-$ (spontaneous radiative decay, $T_1 = 1/\Gamma$).
* $\hat{L}_{\text{deph}} = \sqrt{\frac{\gamma_{\text{pure}}}{2}} \hat{\sigma}_z$ (pure elastic dephasing).

### The Bloch Vector $\vec{u} = (u, v, w)^T$
We parameterize the $2 \times 2$ density matrix $\hat{\rho}$ using the real Bloch vector:
$$\hat{\rho} = \frac{1}{2} \left( \hat{I} + u \hat{\sigma}_x + v \hat{\sigma}_y + w \hat{\sigma}_z \right) = \frac{1}{2} \begin{pmatrix} 1 + w & u - iv \\ u + iv & 1 - w \end{pmatrix}$$
Physical significance of components:
* $u = \rho_{eg} + \rho_{ge} = 2 \mathrm{Re}(\rho_{ge})$: **Dispersive (in-phase) polarization**.
* $v = i(\rho_{eg} - \rho_{ge}) = 2 \mathrm{Im}(\rho_{ge})$: **Absorptive (in-quadrature) polarization**.
* $w = \rho_{ee} - \rho_{gg}$: **Atomic population inversion** ($-1 \le w \le +1$).

Substituting this parameterization yields the celebrated **Optical Bloch Equations (OBEs)**:
$$\dot{u} = -\Delta v - \frac{u}{T_2}$$
$$\dot{v} = \Delta u + \Omega w - \frac{v}{T_2}$$
$$\dot{w} = -\Omega v - \frac{w - w_{\text{eq}}}{T_1}$$
where:
* $T_1 = 1/\Gamma$ is the **longitudinal relaxation time** (population lifetime).
* $T_2$ is the **transverse dephasing time** (coherence lifetime), given by:
  $$\frac{1}{T_2} = \frac{1}{2T_1} + \gamma_{\text{pure}} \implies T_2 \le 2 T_1$$
* $w_{\text{eq}} = -1$ is the thermal equilibrium inversion (at optical frequencies, $k_B T \ll \hbar\omega_0$, so the thermal population is entirely in the ground state $|g\rangle$).


---

## 3. Analytical Steady-State Solutions & Power Broadening

Setting time derivatives to zero ($\dot{u} = \dot{v} = \dot{w} = 0$), the steady-state solutions $(\bar{u}, \bar{v}, \bar{w})$ can be solved algebraically in closed form:

$$\bar{u}(\Delta) = -\frac{\Delta \Omega T_2^2}{1 + (\Delta T_2)^2 + \Omega^2 T_1 T_2} w_{\text{eq}}$$
$$\bar{v}(\Delta) = \frac{\Omega T_2}{1 + (\Delta T_2)^2 + \Omega^2 T_1 T_2} w_{\text{eq}}$$
$$\bar{w}(\Delta) = \frac{1 + (\Delta T_2)^2}{1 + (\Delta T_2)^2 + \Omega^2 T_1 T_2} w_{\text{eq}}$$

Substituting $w_{\text{eq}} = -1$, the steady-state excited state population is:
$$\rho_{ee}^{\text{ss}} = \frac{1 + \bar{w}}{2} = \frac{1}{2} \frac{\Omega^2 T_1 T_2}{1 + \Delta^2 T_2^2 + \Omega^2 T_1 T_2} = \frac{1}{2} \frac{s_0}{1 + s_0 + (\Delta T_2)^2}$$
where $s_0 = \Omega^2 T_1 T_2 = \frac{I}{I_{\text{sat}}}$ is the **on-resonance saturation parameter**.

### Key Physical Consequences:
1. **The Maximum Population Inversion Limit:**
   As the driving laser power increases indefinitely ($s_0 \to \infty$ or $\Omega \gg \Gamma$):
   $$\lim_{\Omega \to \infty} \rho_{ee}^{\text{ss}} = \frac{1}{2}, \quad \lim_{\Omega \to \infty} \bar{w} = 0$$
   *A continuous-wave laser can never achieve population inversion ($w > 0$, $\rho_{ee} > 0.5$) in a two-level atom!* Stimulated emission exactly balances stimulated absorption.
2. **Power Broadening:**
   The absorption spectrum $\mathcal{A}(\Delta) \propto -\bar{v}(\Delta)$ is a Lorentzian whose full-width at half-maximum (FWHM) broadens with driving intensity:
   $$\Delta \omega_{\text{FWHM}} = \frac{2}{T_2} \sqrt{1 + s_0} = \Gamma_0 \sqrt{1 + \frac{I}{I_{\text{sat}}}}$$
   At weak intensity ($s_0 \ll 1$), the linewidth is the natural lifetime-broadened width $2/T_2 = \Gamma$. At strong driving ($s_0 \gg 1$), the transition broadens proportionally to $\Omega \sqrt{T_1/T_2}$.


In [ ]:
# Implementation: Numerical Integrator and Analytical Solvers for the Optical Bloch Equations

class OpticalBlochSimulator:
    """
    Simulates dissipative two-level atom dynamics via the Optical Bloch Equations (OBEs).
    """
    def __init__(self, omega_rabi=1.0, delta=0.0, gamma_decay=1.0, gamma_pure=0.0):
        """
        Parameters:
            omega_rabi: Driving Rabi frequency Omega (in units of Gamma).
            delta: Laser detuning Delta = omega_L - omega_0.
            gamma_decay: Spontaneous emission rate Gamma = 1/T1.
            gamma_pure: Pure dephasing rate gamma_pure.
        """
        self.omega = float(omega_rabi)
        self.delta = float(delta)
        self.gamma = float(gamma_decay)
        self.gamma_pure = float(gamma_pure)
        
        # Relaxation times
        self.T1 = 1.0 / self.gamma
        self.T2 = 1.0 / (0.5 * self.gamma + self.gamma_pure)
        self.w_eq = -1.0  # Ground state equilibrium
        
    def odes(self, t, y):
        """Right-hand side of the Optical Bloch Equations: dy/dt = f(t, y) where y = [u, v, w]."""
        u, v, w = y
        du_dt = -self.delta * v - u / self.T2
        dv_dt =  self.delta * u + self.omega * w - v / self.T2
        dw_dt = -self.omega * v - (w - self.w_eq) / self.T1
        return [du_dt, dv_dt, dw_dt]
        
    def solve(self, t_span=(0.0, 20.0), y0=(0.0, 0.0, -1.0), n_points=1000):
        """Integrates the OBEs from t_span[0] to t_span[1] starting from initial state y0."""
        t_eval = np.linspace(t_span[0], t_span[1], n_points)
        sol = solve_ivp(self.odes, t_span, y0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-10)
        return sol.t, sol.y[0], sol.y[1], sol.y[2]
        
    def analytical_steady_state(self, delta=None):
        """Calculates analytical steady-state values (u_ss, v_ss, w_ss) for a given detuning."""
        d = self.delta if delta is None else delta
        denom = 1.0 + (d * self.T2)**2 + (self.omega**2) * self.T1 * self.T2
        u_ss = -(d * self.omega * self.T2**2 / denom) * self.w_eq
        v_ss = (self.omega * self.T2 / denom) * self.w_eq
        w_ss = ((1.0 + (d * self.T2)**2) / denom) * self.w_eq
        return u_ss, v_ss, w_ss

# Verify consistency of numerical integration with analytical steady-state
test_sim = OpticalBlochSimulator(omega_rabi=3.0, delta=1.0, gamma_decay=1.0, gamma_pure=0.2)
t_arr, u_num, v_num, w_num = test_sim.solve(t_span=(0.0, 40.0))
u_ana, v_ana, w_ana = test_sim.analytical_steady_state()

print("--- Optical Bloch Equation Benchmark Verification ---")
print(f"Numerical Final w(t=40):  {w_num[-1]:.6f}")
print(f"Analytical Steady-State w: {w_ana:.6f}")
print(f"Absolute Inversion Error: {abs(w_num[-1] - w_ana):.3e}")
assert abs(w_num[-1] - w_ana) < 1e-4, "Steady state numerical convergence benchmark failed!"
print("Benchmark Passed: Numerical trajectory converged to analytical steady-state with < 0.01% error.")


In [ ]:
# Publication-Quality 4-Panel Visualization: Dissipative Dynamics, 3D Bloch Sphere, Spectrum & Saturation

fig = plt.figure(figsize=(16, 12))

# =============================================================================
# Panel 1: Time Evolution of Bloch Vector Components (Damped Rabi Oscillations)
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

sim_res = OpticalBlochSimulator(omega_rabi=4.0, delta=0.0, gamma_decay=1.0, gamma_pure=0.0)
t_val, u_val, v_val, w_val = sim_res.solve(t_span=(0.0, 15.0), y0=(0.0, 0.0, -1.0))
u_ss, v_ss, w_ss = sim_res.analytical_steady_state()

ax1.plot(t_val, w_val, 'b-', label=r"Inversion $w(t) = \rho_{ee} - \rho_{gg}$")
ax1.plot(t_val, v_val, 'r--', label=r"Absorption $v(t) = 2\mathrm{Im}(\rho_{ge})$")
ax1.plot(t_val, u_val, 'g-.', label=r"Dispersion $u(t) = 2\mathrm{Re}(\rho_{ge})$")
ax1.axhline(w_ss, color='navy', linestyle=':', alpha=0.7, label=rf"Steady-State $w_{{\mathrm{{ss}}}} = {w_ss:.2f}$")
ax1.set_xlabel(r"Time ($\Gamma t$)")
ax1.set_ylabel("Bloch Components")
ax1.set_title(rf"Damped Rabi Oscillations ($\Omega = 4\Gamma$, $\Delta = 0$)")
ax1.set_ylim(-1.05, 1.05)
ax1.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 2: 3D Bloch Sphere Trajectory (Pure State to Mixed Steady State)
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2, projection='3d')

# Render translucent Bloch sphere wireframe
phi_grid, theta_grid = np.mgrid[0.0:2.0*np.pi:40j, 0.0:np.pi:20j]
xs = np.sin(theta_grid) * np.cos(phi_grid)
ys = np.sin(theta_grid) * np.sin(phi_grid)
zs = np.cos(theta_grid)
ax2.plot_surface(xs, ys, zs, color='silver', alpha=0.15, edgecolor='none')
ax2.plot_wireframe(xs, ys, zs, color='gray', alpha=0.2, rstride=5, cstride=5, linewidth=0.5)

# Render coordinate axes
ax2.plot([-1.1, 1.1], [0, 0], [0, 0], 'k-', lw=0.8, alpha=0.5)
ax2.plot([0, 0], [-1.1, 1.1], [0, 0], 'k-', lw=0.8, alpha=0.5)
ax2.plot([0, 0], [0, 0], [-1.1, 1.1], 'k-', lw=0.8, alpha=0.5)

# Plot trajectory with gradient color
p = ax2.scatter(u_val, v_val, w_val, c=t_val, cmap='plasma', s=8, alpha=0.8, label="Trajectory")
ax2.plot([0, 0], [0, 0], [-1, -1], 'ko', markersize=7, label="Initial $|g\\rangle$ (South Pole)")
ax2.plot([u_ss], [v_ss], [w_ss], 'r*', markersize=12, label="Steady State Inside Ball")

ax2.set_xlabel("$u$ (Dispersion)", labelpad=6)
ax2.set_ylabel("$v$ (Absorption)", labelpad=6)
ax2.set_zlabel("$w$ (Inversion)", labelpad=6)
ax2.set_title("3D Bloch Trajectory Spiraling into Mixed State")
ax2.legend(loc="upper left", fontsize=8, frameon=True)
ax2.view_init(elev=20, azim=45)

# =============================================================================
# Panel 3: Absorption & Dispersion Spectra vs. Detuning (Power Broadening)
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

delta_range = np.linspace(-10.0, 10.0, 500)
rabi_powers = [0.5, 1.5, 3.0, 6.0]

for om in rabi_powers:
    sim_scan = OpticalBlochSimulator(omega_rabi=om, delta=0.0, gamma_decay=1.0)
    abs_profile = [-sim_scan.analytical_steady_state(d)[1] for d in delta_range]
    fwhm = 2.0 * np.sqrt(1.0 + (om**2 * 2.0))  # T1=1, T2=2 -> s0 = 2*om^2
    ax3.plot(delta_range, abs_profile, label=rf"$\Omega = {om:.1f}\Gamma$ (FWHM $\approx {fwhm:.1f}\Gamma$)")

ax3.set_xlabel(r"Detuning $\Delta/\Gamma = (\omega_L - \omega_0)/\Gamma$")
ax3.set_ylabel(r"Absorption Component $-v_{\mathrm{ss}}$")
ax3.set_title("Absorption Spectrum: Lorentzian Profile & Power Broadening")
ax3.legend(frameon=True)

# =============================================================================
# Panel 4: Saturation of Excited State Population (The 0.5 Asymptotic Limit)
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

omega_scan = np.logspace(-1, 1.8, 200)
rho_ee_pure0 = []
rho_ee_deph1 = []
rho_ee_deph5 = []

for om in omega_scan:
    # Pure radiative: gamma_pure = 0 -> T2 = 2 T1
    s_rad = OpticalBlochSimulator(omega_rabi=om, delta=0.0, gamma_decay=1.0, gamma_pure=0.0)
    rho_ee_pure0.append(0.5 * (1.0 + s_rad.analytical_steady_state(0.0)[2]))
    
    # Moderate dephasing: gamma_pure = 1.0 -> 1/T2 = 0.5 + 1.0 = 1.5
    s_dep1 = OpticalBlochSimulator(omega_rabi=om, delta=0.0, gamma_decay=1.0, gamma_pure=1.0)
    rho_ee_deph1.append(0.5 * (1.0 + s_dep1.analytical_steady_state(0.0)[2]))
    
    # Strong dephasing: gamma_pure = 5.0
    s_dep5 = OpticalBlochSimulator(omega_rabi=om, delta=0.0, gamma_decay=1.0, gamma_pure=5.0)
    rho_ee_deph5.append(0.5 * (1.0 + s_dep5.analytical_steady_state(0.0)[2]))

ax4.semilogx(omega_scan, rho_ee_pure0, 'b-', label=r"Radiative Limit ($T_2 = 2T_1$)")
ax4.semilogx(omega_scan, rho_ee_deph1, 'g--', label=r"Moderate Dephasing ($\gamma_{\mathrm{pure}} = 1.0\Gamma$)")
ax4.semilogx(omega_scan, rho_ee_deph5, 'r:', label=r"Strong Dephasing ($\gamma_{\mathrm{pure}} = 5.0\Gamma$)")
ax4.axhline(0.5, color='k', linestyle='--', alpha=0.7, label=r"Absolute Saturation Limit $\rho_{ee} = 0.5$")

ax4.set_xlabel(r"Rabi Frequency $\Omega/\Gamma$")
ax4.set_ylabel(r"Steady-State Population $\rho_{ee}^{\mathrm{ss}}$")
ax4.set_title(r"Optical Saturation Curve: $\rho_{ee} \to 0.5$ Asymptote")
ax4.set_ylim(-0.02, 0.58)
ax4.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Can't a Monochromatic CW Laser Achieve Population Inversion in a Two-Level Atom?**
>
> In laser physics, population inversion ($\rho_{ee} > \rho_{gg}$, or $w > 0$) is the prerequisite for optical gain and lasing. One might intuitively expect that dialing up a drive laser to extreme power ($\Omega \gg \Gamma$) would pump all atoms into the excited state $|e\rangle$.
>
> However, Einstein's $A$ and $B$ coefficient relation proves that the rate of stimulated absorption $|g\rangle \to |e\rangle$ is **strictly equal** to the rate of stimulated emission $|e\rangle \to |g\rangle$ for non-degenerate levels ($B_{12} = B_{21}$). Under intense continuous-wave drive, the stimulated rates dwarf spontaneous emission ($\Omega^2 \gg \Gamma^2$), locking the two levels into equal populations:
> $$\lim_{\Omega \to \infty} \rho_{ee}^{\text{ss}} = \lim_{\Omega \to \infty} \rho_{gg}^{\text{ss}} = \frac{1}{2} \implies w_{\text{ss}} \to 0$$
> To achieve true population inversion ($w > 0$), one must break this symmetry—either dynamically using ultrafast resonant $\pi$-pulses (Notebook 3.2), or by using three- or four-level pumping architectures where fast non-radiative decay breaks detailed balance.


In [ ]:
# Interactive Exploration: Optical Bloch Dynamics Parameter Sweep

def plot_interactive_obe(omega=3.0, delta=0.0, gamma_pure=0.0, t_max=20.0):
    """
    Renders the dynamic Bloch vector trajectory and population inversion
    for user-selected physical parameters.
    """
    sim = OpticalBlochSimulator(omega_rabi=omega, delta=delta, gamma_decay=1.0, gamma_pure=gamma_pure)
    t_span = (0.0, t_max)
    t_ev, u_ev, v_ev, w_ev = sim.solve(t_span=t_span)
    u_s, v_s, w_s = sim.analytical_steady_state()
    
    # Calculate purity of the state over time: Tr(rho^2) = (1 + |u|^2 + |v|^2 + |w|^2) / 2
    r_sq = u_ev**2 + v_ev**2 + w_ev**2
    purity = 0.5 * (1.0 + r_sq)
    
    fig, (ax_dyn, ax_pur) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Population dynamics
    ax_dyn.plot(t_ev, w_ev, 'b-', label=r"Inversion $w(t)$")
    ax_dyn.plot(t_ev, v_ev, 'r--', label=r"Coherence $v(t)$ (Absorption)")
    ax_dyn.axhline(w_s, color='navy', linestyle=':', label=rf"Analytical $w_{{\mathrm{{ss}}}} = {w_s:.3f}$")
    ax_dyn.set_xlabel(r"Time ($\Gamma t$)")
    ax_dyn.set_ylabel("Bloch Vector Component")
    ax_dyn.set_title(rf"OBE Dynamics ($\Omega={omega:.1f}\Gamma$, $\Delta={delta:.1f}\Gamma$, $\gamma_{{\mathrm{{pure}}}}={gamma_pure:.1f}\Gamma$)")
    ax_dyn.set_ylim(-1.05, 1.05)
    ax_dyn.legend(frameon=True)
    
    # Quantum state purity
    ax_pur.plot(t_ev, purity, 'purple', label=r"Purity $\mathcal{P}(t) = \mathrm{Tr}(\hat{\rho}^2)$")
    ax_pur.axhline(0.5, color='gray', linestyle='--', label=r"Max Mixed State $\mathcal{P} = 0.5$")
    ax_pur.axhline(1.0, color='gold', linestyle='--', label=r"Pure State $\mathcal{P} = 1.0$")
    ax_pur.set_xlabel(r"Time ($\Gamma t$)")
    ax_pur.set_ylabel("Quantum State Purity")
    ax_pur.set_title("Decoherence & Loss of Purity")
    ax_pur.set_ylim(0.45, 1.05)
    ax_pur.legend(frameon=True)
    
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_obe,
        omega=widgets.FloatSlider(value=3.0, min=0.0, max=10.0, step=0.2, description=r"$\Omega/\Gamma$:"),
        delta=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.2, description=r"$\Delta/\Gamma$:"),
        gamma_pure=widgets.FloatSlider(value=0.0, min=0.0, max=4.0, step=0.2, description=r"$\gamma_{\mathrm{pure}}/\Gamma$:"),
        t_max=widgets.FloatSlider(value=20.0, min=5.0, max=40.0, step=5.0, description=r"$t_{\mathrm{max}}$:")
    )
else:
    print("Static exploration demonstration (default parameters: Omega = 3.0, Delta = 0.0, gamma_pure = 0.5):")
    plot_interactive_obe(omega=3.0, delta=0.0, gamma_pure=0.5, t_max=20.0)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Radiative Limit of Transverse Dephasing
Prove analytically that in the pure radiative limit (no elastic dephasing, $\gamma_{\text{pure}} = 0$), the coherence decay rate $\gamma_{\perp} = 1/T_2$ is exactly half the population decay rate $\Gamma = 1/T_1$:
$$T_2 = 2 T_1$$

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

The off-diagonal density matrix element is given by $\rho_{ge}(t) = \langle g|\hat{\rho}(t)|e\rangle$. In the absence of driving, the state vector under spontaneous decay evolves as:
$$|\psi(t)\rangle = c_g(t)|g\rangle + c_e(0)e^{-i\omega_0 t}e^{-\Gamma t / 2}|e\rangle$$
The excited state population decays with rate $\Gamma$:
$$\rho_{ee}(t) = |c_e(t)|^2 = |c_e(0)|^2 e^{-\Gamma t} \implies T_1 = \frac{1}{\Gamma}$$
The coherence $\rho_{ge}(t) = c_g^*(t) c_e(t)$ decays as the probability amplitude:
$$|\rho_{ge}(t)| \propto |c_e(t)| = |c_e(0)| e^{-\Gamma t / 2} = |c_e(0)| e^{-t / T_2}$$
Matching exponents:
$$\frac{1}{T_2} = \frac{\Gamma}{2} = \frac{1}{2T_1} \implies T_2 = 2T_1$$
Pure dephasing $\gamma_{\text{pure}}$ adds extra phase randomization without changing populations, resulting in the general inequality $T_2 \le 2T_1$.
</details>

---

### Challenge 2: Power Broadening Numerical Verification
For a radiatively lifetime-limited atom ($T_1 = 1$, $T_2 = 2$) driven at Rabi frequency $\Omega = 4\Gamma$, compute the expected theoretical FWHM of the absorption spectrum and compare it against the numerically computed full-width at half-maximum.


In [ ]:
# Numerical Verification of Challenge 2: Analytical vs. Numerical FWHM

om_test = 4.0
T1_test = 1.0
T2_test = 2.0

# On-resonance saturation parameter
s0_test = om_test**2 * T1_test * T2_test  # 4^2 * 1 * 2 = 32

# Theoretical FWHM
gamma_0 = 2.0 / T2_test  # = 1.0
fwhm_theory = gamma_0 * np.sqrt(1.0 + s0_test)

# Numerical scan across detuning
delta_dense = np.linspace(-15.0, 15.0, 20000)
sim_test = OpticalBlochSimulator(omega_rabi=om_test, delta=0.0, gamma_decay=1.0/T1_test, gamma_pure=0.0)

abs_curve = np.array([-sim_test.analytical_steady_state(d)[1] for d in delta_dense])
half_max = np.max(abs_curve) / 2.0

# Find indices where absorption exceeds half maximum
indices_above_half = np.where(abs_curve >= half_max)[0]
fwhm_num = delta_dense[indices_above_half[-1]] - delta_dense[indices_above_half[0]]

print(f"--- Power Broadening Verification (Omega = {om_test} Gamma) ---")
print(f"Saturation Parameter s0:   {s0_test:.1f}")
print(f"Theoretical FWHM:          {fwhm_theory:.4f} Gamma")
print(f"Numerical Scanned FWHM:    {fwhm_num:.4f} Gamma")
print(f"Discrepancy:               {abs(fwhm_num - fwhm_theory):.2e} Gamma")

assert abs(fwhm_num - fwhm_theory) < 0.01, "Power broadening FWHM check failed!"
print("Verification Passed: Numerical linewidth matches theoretical power broadening formula to < 0.1% accuracy!")


---

## 5. Summary & Key Formulas

| Concept | Mathematical Expression | Physical Interpretation |
| :--- | :--- | :--- |
| **RWA Hamiltonian** | $\hat{H}_{\text{RWA}} = \frac{\hbar}{2} (-\Delta \hat{\sigma}_z + \Omega \hat{\sigma}_x)$ | Atom-field coupling in the laser rotating frame |
| **Bloch Vector** | $\hat{\rho} = \frac{1}{2}(\hat{I} + u\hat{\sigma}_x + v\hat{\sigma}_y + w\hat{\sigma}_z)$ | Mapping quantum state to the unit 3D ball |
| **Decoherence Times** | $\frac{1}{T_2} = \frac{1}{2T_1} + \gamma_{\text{pure}}$ | Transverse dephasing bounded by longitudinal decay ($T_2 \le 2T_1$) |
| **Steady-State Inversion** | $\bar{w}(\Delta) = \frac{1 + (\Delta T_2)^2}{1 + (\Delta T_2)^2 + \Omega^2 T_1 T_2} w_{\text{eq}}$ | Continuous saturation; bounded by $\rho_{ee} \to 0.5$ as $\Omega \to \infty$ |
| **Power Broadening** | $\Delta\omega_{\text{FWHM}} = \frac{2}{T_2}\sqrt{1 + s_0}$ | Intensity-dependent spectral broadening |

---

### Further Reading & Foundational Literature
* **L. Allen & J. H. Eberly**, *Optical Resonance and Two-Level Atoms*, Dover Publications (1987).
* **C. Cohen-Tannoudji, J. Dupont-Roc, & G. Grynberg**, *Atom-Photon Interactions: Basic Processes and Applications*, Wiley-VCH (1998).
* **M. Scully & M. S. Zubairy**, *Quantum Optics*, Cambridge University Press (1997), Chapter 5.

---

### Next Notebook in Module 3
In **Notebook 3.2: Rabi Oscillations, Detuning & Pulse Control**, we transition from continuous-wave steady-state driving to coherent pulsed manipulation:
* Resonant and off-resonant Rabi flopping with generalized Rabi frequency $\Omega_{\text{eff}} = \sqrt{\Omega^2 + \Delta^2}$.
* Implementing precision unitary gates: $\pi$-pulses for complete state inversion and $\pi/2$-pulses for equal superposition creation.
* **Ramsey fringes & interferometry**: The cornerstone of atomic clocks, quantum sensing, and precision magnetometry.
